### Load up selenium and the Chrome webdriver

In [1]:
# uncomment the selenium installation line if necessary
%pip install selenium
import selenium
from selenium import webdriver
from selenium.common import NoSuchElementException, ElementNotInteractableException
from selenium.webdriver.common.by import By
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

  Using cached certifi-2025.1.31-py3-none-any.whl.metadata (2.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 10.8 MB/s eta 0:00:00a 0:00:01
Using cached certifi-2025.1.31-py3-none-any.whl (166 kB)
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
Note: you may need to restart the kernel to use updated packages.


In [2]:
driver = webdriver.Chrome()
#driver = webdriver.Firefox()  


In [3]:
driver.get('https://generalssb-prod.ec.njit.edu/BannerExtensibility/customPage/page/stuRegCrseSched')

### In the left navigation, find the 16th entry ("CS") and click on it

#### This will cause the main content area to load all of the CS courses

In [4]:
driver.implicitly_wait(5)
left_link = driver.find_element(By.ID, 'pbid-subjListTableSubjectLink-16')

ProtocolError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [ ]:
cs_link = left_link.find_element(by=By.TAG_NAME, value='a')
cs_link.click()

In [ ]:
my_span = driver.find_element(by=By.ID, value='pbid-courseListSectionDetailSections-0')

In [ ]:
print(my_span)

<selenium.webdriver.remote.webelement.WebElement (session="133ff4450084c8fc78551482fe4dca5c", element="f.5103C79310F36999D952222225E36128.d.B06A8E3A632FF5284ED201869ED8115D.e.46")>


In [ ]:
course_headers = my_span.find_elements(by=By.TAG_NAME, value="H4")

In [ ]:
course_tables = my_span.find_elements(by=By.TAG_NAME, value="table")

In [ ]:
print("I have {} headers and {} tables".format(len(course_headers), len(course_tables)))

I have 78 headers and 78 tables


In [ ]:
def create_column_headings(table_headers):
    loc_columns = ['Course_name']
    for i in range(len(table_headers)):
        loc_columns.append(table_headers[i].text)
    return loc_columns

In [ ]:
def create_course_listings(c_headers, c_tables):
    c_list = []
    for i in range(len(c_headers)):
        course_name = c_headers[i].text
#       collect the rows for each section of the course
        t_rows = c_tables[i].find_elements(by=By.CLASS_NAME, value='success')
#       for each row, get all of the cells
        for k in range(len(t_rows)):
            course_row = []
            course_row.append(course_name)
            t_cells = t_rows[k].find_elements(by=By.TAG_NAME, value='td')
            for l in range(len(t_cells)):
                course_row.append(t_cells[l].text)
            c_list.append(course_row)
    return c_list


In [ ]:
# create column headings by using the 'th' cells of the first table
t_headers = course_tables[0].find_elements(by=By.TAG_NAME, value="th")
columns = create_column_headings(t_headers)

In [ ]:
# use the list of course_headers and the list of course_tables to build
#   individual rows - one for each course/section
course_list = create_course_listings(course_headers, course_tables)
df = pd.DataFrame(course_list, columns=columns)
driver.quit()

In [ ]:
df

,Course_name,Section,CRN,Days,Times,Location,Status,Max,Now,Instructor,Delivery Mode,Credits,Info,Comments
0,CS 100 - ROADMAP TO COMPUTING,003,91875,TF,2:30 PM - 3:50 PM,CKB 217,Open,7,1,"Spirollari, Junilda",Face-to-Face,3,Book,This course has common exams.\nSee https://www...
1,CS 100 - ROADMAP TO COMPUTING,005,91876,TF,1:00 PM - 2:20 PM,CKB 217,Open,7,5,"Spirollari, Junilda",Face-to-Face,3,Book,This course has common exams.\nSee https://www...
2,CS 100 - ROADMAP TO COMPUTING,007,91877,MW,10:00 AM - 11:20 AM,CKB 217,Open,7,3,"Qerimaj, Jertishta",Face-to-Face,3,Book,This course has common exams.\nSee https://www...
3,CS 100 - ROADMAP TO COMPUTING,009,91878,MR,2:30 PM - 3:50 PM,CKB 217,Open,7,4,"Qerimaj, Jertishta",Face-to-Face,3,Book,This course has common exams.\nSee https://www...
4,CS 100 - ROADMAP TO COMPUTING,011,91879,TR,4:00 PM - 5:20 PM,CKB 217,Open,7,0,"Qerimaj, Jertishta",Face-to-Face,3,Book,This course has common exams.\nSee https://www...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
436,CS 792 - PRE-DOCTORAL RESEARCH,623,92364,,TBA,,Open,19,0,"Theodoratos, Dimitrios",Face-to-Face,3,Book,See department for permission to enroll.
437,CS 792 - PRE-DOCTORAL RESEARCH,624,92365,,TBA,,Open,19,0,"Wang, Guiling",Face-to-Face,3,Book,See department for permission to enroll.
438,CS 792 - PRE-DOCTORAL RESEARCH,625,92366,,TBA,,Open,19,0,"Mili, Ali",Face-to-Face,3,Book,See department for permission to enroll.
439,CS 792 - PRE-DOCTORAL RESEARCH,626,92367,,TBA,,Open,19,0,"Chakareski, Jakov",Face-to-Face,3,Book,See department for permission to enroll.


In [ ]:
df.describe()

,Course_name,Section,CRN,Days,Times,Location,Status,Max,Now,Instructor,Delivery Mode,Credits,Info,Comments
count,441,441,441,441,441,441,441,441,441,441,441,441,441,441
unique,58,70,441,13,16,46,1,19,33,80,3,3,1,12
top,CS 792 - PRE-DOCTORAL RESEARCH,001,91875,,TBA,,Open,19,0,,Face-to-Face,3,Book,See department for permission to enroll.
freq,47,23,1,345,344,352,441,331,329,13,428,400,441,329


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 441 entries, 0 to 440
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Course_name    441 non-null    object
 1   Section        441 non-null    object
 2   CRN            441 non-null    object
 3   Days           441 non-null    object
 4   Times          441 non-null    object
 5   Location       441 non-null    object
 6   Status         441 non-null    object
 7   Max            441 non-null    object
 8   Now            441 non-null    object
 9   Instructor     441 non-null    object
 10  Delivery Mode  441 non-null    object
 11  Credits        441 non-null    object
 12  Info           441 non-null    object
 13  Comments       441 non-null    object
dtypes: object(14)
memory usage: 48.4+ KB


In [ ]:
df['Max'] = df['Max'].astype('int')
df['Now'] = df['Now'].astype('int')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 441 entries, 0 to 440
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Course_name    441 non-null    object
 1   Section        441 non-null    object
 2   CRN            441 non-null    object
 3   Days           441 non-null    object
 4   Times          441 non-null    object
 5   Location       441 non-null    object
 6   Status         441 non-null    object
 7   Max            441 non-null    int64 
 8   Now            441 non-null    int64 
 9   Instructor     441 non-null    object
 10  Delivery Mode  441 non-null    object
 11  Credits        441 non-null    object
 12  Info           441 non-null    object
 13  Comments       441 non-null    object
dtypes: int64(2), object(12)
memory usage: 48.4+ KB


In [ ]:
df = df[~df['Course_name'].str.contains('CS 488')]
df = df[~df['Course_name'].str.contains('CS 700B')]
df = df[~df['Course_name'].str.contains('CS 701B')]
df = df[~df['Course_name'].str.contains('CS 725')]
df = df[~df['Course_name'].str.contains('CS 792')]
df = df[~df['Course_name'].str.contains('CS 790A')]
df = df[~df['Course_name'].str.contains('CS 726')]

df

,Course_name,Section,CRN,Days,Times,Location,Status,Max,Now,Instructor,Delivery Mode,Credits,Info,Comments
0,CS 100 - ROADMAP TO COMPUTING,003,91875,TF,2:30 PM - 3:50 PM,CKB 217,Open,7,1,"Spirollari, Junilda",Face-to-Face,3,Book,This course has common exams.\nSee https://www...
1,CS 100 - ROADMAP TO COMPUTING,005,91876,TF,1:00 PM - 2:20 PM,CKB 217,Open,7,5,"Spirollari, Junilda",Face-to-Face,3,Book,This course has common exams.\nSee https://www...
2,CS 100 - ROADMAP TO COMPUTING,007,91877,MW,10:00 AM - 11:20 AM,CKB 217,Open,7,3,"Qerimaj, Jertishta",Face-to-Face,3,Book,This course has common exams.\nSee https://www...
3,CS 100 - ROADMAP TO COMPUTING,009,91878,MR,2:30 PM - 3:50 PM,CKB 217,Open,7,4,"Qerimaj, Jertishta",Face-to-Face,3,Book,This course has common exams.\nSee https://www...
4,CS 100 - ROADMAP TO COMPUTING,011,91879,TR,4:00 PM - 5:20 PM,CKB 217,Open,7,0,"Qerimaj, Jertishta",Face-to-Face,3,Book,This course has common exams.\nSee https://www...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,CS 684 - SOFTWARE TEST & QUAL ASSURANCE,001,92115,F,8:30 AM - 11:20 AM,CKB 114,Open,30,11,,Face-to-Face,3,Book,
188,CS 698 - ST: GPU CLUSTER PROGRAMMING,001,92116,MW,10:00 AM - 11:20 AM,GITC 1100,Open,30,5,"Sohn, Andrew",Face-to-Face,3,Book,
351,CS 732 - ADVANCED MACHINE LEARNING,101,92279,W,6:00 PM - 8:50 PM,,Open,30,3,"Houle, Michael",Synchronous Online,3,Book,Synchronous Online Course\nhttps://www.njit.ed...
352,CS 785 - ST: ALGORITHMS FOR ENABLING RESPONSIB...,001,92280,TR,1:00 PM - 2:20 PM,CKB 207,Open,30,1,"Basu Roy, Senjuti",Face-to-Face,3,Book,
